# Participant-level CGM summaries

This notebook calculates participant-level continuous glucose monitoring (CGM) summaries after excluding the two-hour windows following standardized meals. The exported table is used to characterize glucose-response traits in analysis notebook 04.

## Notebook flow

1. Load participant metadata and CGM measurements, align participant keys, and convert glucose from mmol/L to mg/dL.
2. Remove invalid readings and exclude CGM measurements within two hours after retained standardized meals.
3. Define glucose-level, variability and glycemic-risk metrics, including time and percentage in or outside the 70–180 mg/dL range.
4. Calculate the metrics separately for each participant and export the summary table.

## Inputs

Source files in `Data/raw data/`: `metadata.csv` and `cgm_data.csv`. Standardized-meal timestamps come from `Data/meal_ppgr.csv` or its ZIP fallback, produced by preprocessing notebook 02. The notebook also requires the `cgmquantify` package.

## Outputs

| File | Contents |
| --- | --- |
| `Data/cgm_metrics.csv` | Participant keys, observation counts and date ranges, glucose-level and variability summaries, range metrics, glycemic-risk indices, and diagnostic fields for unavailable metrics. |

Time in and outside range is expressed in minutes using a nominal 15-minute sampling interval; the corresponding percentages are stored separately. Record counts and a preview of the participant summaries are displayed in the notebook.

## 1. Setup

Resolve project paths, import packages and configure warning handling.

In [1]:
from pathlib import Path
import sys

# Find the project when launched from its root, Code/, or a notebook subfolder.
for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    CODE_DIR = PROJECT_ROOT / "Code"
    if (CODE_DIR / "data_paths.py").is_file():
        break
else:
    raise FileNotFoundError("Open this notebook from within the project directory.")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Shared defaults; override individual paths here if needed.
from data_paths import DATA_DIR, METADATA_PATH, MEAL_DATA_PATH, CGM_METRICS_PATH
from data_paths import (
    RAW_DATA_DIR, RAW_METADATA_PATH, RAW_CGM_PATH, RAW_FOOD_PATH,
    MERGED_MEALS_PATH, CLEANED_FOOD_PATH,
)

import warnings
warnings.filterwarnings(
    'ignore',
    message='A value is trying to be set on a copy of a DataFrame',
)
warnings.filterwarnings(
    'ignore',
    message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated',
)
warnings.filterwarnings("ignore")


warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="cgmquantify"
)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings(
    "ignore",
    message=r"The behavior of DataFrame\.std with axis=None is deprecated.*",
    category=FutureWarning,
)

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from datetime import datetime, timedelta

from scipy.signal import argrelmin

pd.set_option('mode.chained_assignment', None)


## 2. Load and prepare CGM measurements

Map participant keys, parse timestamps, convert glucose units and remove invalid measurements.

In [2]:
meta_data = pd.read_csv(RAW_METADATA_PATH).reset_index(drop=True)

meta_data.rename(columns={'subject_app_key':'subject_key'}, inplace=True)

if 'id' in meta_data.columns:
    id_to_subjectKey = dict(zip(meta_data['id'], meta_data['subject_key']))
    meta_data.rename(columns={'id':'fay-id'}, inplace=True)
else:
    id_to_subjectKey = dict(zip(meta_data['fay-id'], meta_data['subject_key']))

In [3]:
import pandas as pd
import numpy as np

df_gluc = (
    pd.read_csv(RAW_CGM_PATH, index_col=0)
    .reset_index(drop=False)
)

df_gluc = df_gluc.rename(
    columns={
        "read_at": "time",
        "user_id": "id",
        "val": "gl_mmol",
    }
)

# Map participant identifiers
df_gluc["subject_key"] = df_gluc["id"].map(id_to_subjectKey)

# Keep only participants with a valid subject_key
df_gluc = df_gluc.dropna(subset=["subject_key"]).copy()

# Ensure glucose is numeric
df_gluc["gl_mmol"] = pd.to_numeric(
    df_gluc["gl_mmol"],
    errors="coerce"
)

# Parse datetime
df_gluc["Time"] = pd.to_datetime(
    df_gluc["time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Convert mmol/L to mg/dL
df_gluc["Glucose"] = df_gluc["gl_mmol"] * 18.0

# Remove invalid timestamps and glucose measurements
df_gluc = df_gluc.dropna(
    subset=["Time", "Glucose"]
)

# Add the Day column required by cgmquantify
df_gluc["Day"] = df_gluc["Time"].dt.date

# Exclude non-positive glucose values.
df_gluc = df_gluc[df_gluc["Glucose"] > 0]

# Sort within participants
df_gluc = df_gluc.sort_values(
    ["subject_key", "Time"]
).reset_index(drop=True)

print(df_gluc.shape)

df_gluc[
    ["subject_key", "Time", "Day", "Glucose"]
].head()

(1522406, 8)


,subject_key,Time,Day,Glucose
0,02ae3856ca04,2018-11-26 08:00:00,2018-11-26,109.08
1,02ae3856ca04,2018-11-26 08:15:00,2018-11-26,135.90
2,02ae3856ca04,2018-11-26 08:30:00,2018-11-26,153.18
3,02ae3856ca04,2018-11-26 08:45:00,2018-11-26,133.02
4,02ae3856ca04,2018-11-26 09:00:00,2018-11-26,122.40


## 3. Exclude standardized-meal windows

Remove CGM observations within two hours after standardized meals identified in the prepared meal table.

In [4]:
df_food = pd.read_csv(MEAL_DATA_PATH, index_col=0)
df_food["eaten_at"] = pd.to_datetime(df_food["eaten_at"])

meals = (df_food.loc[df_food.is_standardized_meal == 1, ['subject_key', 'eaten_at']]
         .dropna()
         .sort_values('eaten_at'))
meals['meal_time'] = meals['eaten_at']

g = df_gluc.sort_values('Time')

m = pd.merge_asof(
    g, meals,
    left_on='Time', right_on='eaten_at',
    by='subject_key',
    tolerance=pd.Timedelta('2h'),
    direction='backward',
)

dropped = m['meal_time'].notna().to_numpy()      # True = inside a 2h post-meal window
df_gluc_clean = g[~dropped]

print(f"before:  {len(g):,}")
print(f"dropped: {dropped.sum():,} ({dropped.mean():.2%})")
print(f"after:   {len(df_gluc_clean):,}")

before:  1,522,406
dropped: 39,586 (2.60%)
after:   1,482,820


## 4. Define participant-level CGM metrics

Specify summary, variability and range metrics, including handling for unavailable calculations.

In [5]:
import pandas as pd
import numpy as np
import cgmquantify as cgm


def safe_intradaysd(data):
    daily_sd = (
        data.groupby("Day")["Glucose"]
        .apply(lambda glucose: np.std(glucose.to_numpy(), ddof=0))
        .to_numpy()
    )

    return (
        np.mean(daily_sd),
        np.median(daily_sd),
        np.std(daily_sd, ddof=0),
    )

def has_repeated_daily_times(data, minimum_repeated_times=1):
    """
    Check whether at least one clock-time has observations on 2+ days.
    """
    minute_of_day = data["Time"].dt.hour * 60 + data["Time"].dt.minute

    repeated_times = minute_of_day.value_counts().ge(2).sum()

    return repeated_times >= minimum_repeated_times

def safe_modd(data):
    if data["Day"].nunique() < 2:
        return np.nan

    if not has_repeated_daily_times(data):
        return np.nan

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Mean of empty slice",
            category=RuntimeWarning,
        )
        return cgm.MODD(data.copy())


def safe_conga24(data):
    if data["Day"].nunique() < 2:
        return np.nan

    if not has_repeated_daily_times(data):
        return np.nan

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Mean of empty slice",
            category=RuntimeWarning,
        )
        return cgm.CONGA24(data.copy())

def TIR(df, sr=15, lower=70, upper=180):
    """Time in range (minutes)."""
    return ((df["Glucose"] >= lower) & (df["Glucose"] <= upper)).sum() * sr


def TOR(df, sr=15, lower=70, upper=180):
    """Time outside range (minutes)."""
    return ((df["Glucose"] < lower) | (df["Glucose"] > upper)).sum() * sr

def PIR(df, lower=70, upper=180):
    """Percent time in range."""
    return ((df["Glucose"] >= lower) & (df["Glucose"] <= upper)).mean() * 100


def POR(df, lower=70, upper=180):
    """Percent time outside range."""
    return ((df["Glucose"] < lower) | (df["Glucose"] > upper)).mean() * 100

In [6]:
def compute_cgm_metrics(participant_df, sampling_rate=15):
    data = (
        participant_df[["Time", "Glucose", "Day"]]
        .dropna(subset=["Time", "Glucose"])
        .sort_values("Time")
        .drop_duplicates(subset=["Time"])
        .reset_index(drop=True)
        .copy()
    )

    metrics = {
        "n_observations": len(data),
        "start_time": data["Time"].min(),
        "end_time": data["Time"].max(),
        "n_days": data["Day"].nunique(),
    }

    if data.empty:
        return pd.Series(metrics)

    metric_functions = {
        "summary": lambda x: cgm.summary(x),
        "interdaysd": lambda x: cgm.interdaysd(x),
        "interdaycv": lambda x: cgm.interdaycv(x),
        "intradaysd": safe_intradaysd,
        "intradaycv": lambda x: cgm.intradaycv(x),
        #"TOR": lambda x: cgm.TOR(x, sr=sampling_rate),
        #"TIR": lambda x: cgm.TIR(x, sr=sampling_rate),
        #"POR": lambda x: cgm.POR(x, sr=sampling_rate),
        "TIR": lambda x: TIR(x, sr=sampling_rate),
        "TOR": lambda x: TOR(x, sr=sampling_rate),
        "PIR": lambda x: PIR(x),
        "POR": lambda x: POR(x),
        "MGE": lambda x: cgm.MGE(x),
        "MGN": lambda x: cgm.MGN(x),
        "MAGE": lambda x: cgm.MAGE(x),
        "J_index": lambda x: cgm.J_index(x),
        "LBGI": lambda x: cgm.LBGI(x),
        "HBGI": lambda x: cgm.HBGI(x),
        "ADRR": lambda x: cgm.ADRR(x),
        "MODD": safe_modd,
        "CONGA24": safe_conga24,
        "GMI": lambda x: cgm.GMI(x),
        "eA1c": lambda x: cgm.eA1c(x),
    }

    for metric_name, metric_function in metric_functions.items():
        try:
            result = metric_function(data)

            if metric_name == "summary":
                metrics.update(dict(zip(
                    ["mean_glucose", "median_glucose", "min_glucose",
                     "max_glucose", "q1_glucose", "q3_glucose"],
                    result
                )))

            elif metric_name == "intradaysd":
                metrics.update(dict(zip(
                    ["intradaysd_mean", "intradaysd_median", "intradaysd_sd"],
                    result
                )))

            elif metric_name == "intradaycv":
                metrics.update(dict(zip(
                    ["intradaycv_mean", "intradaycv_median", "intradaycv_sd"],
                    result
                )))
            else:
                metrics[metric_name] = result

        except Exception as error:
            metrics[f"{metric_name}_error"] = str(error)

    return pd.Series(metrics)

## 5. Calculate and export participant summaries

Calculate metrics in parallel, display a preview and save `Data/cgm_metrics.csv`.

In [7]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import pandas as pd
import os
warnings.filterwarnings("ignore", category=FutureWarning, message=".*DataFrame.std with axis=None.*")

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module=r"numpy\.core\.fromnumeric"
)

def process_subject(subject_key, group, sampling_rate=15):
    metrics = compute_cgm_metrics(
        group,
        sampling_rate=sampling_rate
    )

    # Convert to dict so the output is easy to combine
    if isinstance(metrics, pd.Series):
        metrics = metrics.to_dict()

    metrics["subject_key"] = subject_key
    return metrics


groups = list(
    df_gluc_clean.groupby("subject_key", sort=False)
)

n_jobs = max(1, os.cpu_count() - 1)

results = Parallel(
    n_jobs=n_jobs,
    backend="loky",
)(
    delayed(process_subject)(
        subject_key,
        group,
        sampling_rate=15
    )
    for subject_key, group in tqdm(
        groups,
        desc="Submitting subjects"
    )
)

metrics_df = (
    pd.DataFrame(results)
    .set_index("subject_key")
    .sort_index().reset_index()
)

metrics_df.head()

Submitting subjects:   0%|          | 0/1010 [00:00<?, ?it/s]

,subject_key,n_observations,start_time,end_time,n_days,mean_glucose,median_glucose,min_glucose,max_glucose,q1_glucose,...,MGN_error,MAGE,J_index,LBGI,HBGI,ADRR,MODD,CONGA24,GMI,eA1c
0,02ae3856ca04,1300,2018-11-26 08:00:00,2018-12-10 06:45:00,15,95.201169,94.77,44.10,183.96,83.88,...,unsupported operand type(s) for +: 'Timestamp'...,9.542,12.268841,1.955364,0.062971,10.168173,12.106015,3.800416,5.587212,4.944292
1,223h73,1386,2019-08-21 06:15:00,2019-11-13 03:00:00,17,115.540779,109.80,70.02,219.78,102.78,...,unsupported operand type(s) for +: 'Timestamp'...,22.749,18.305420,0.277840,0.608166,8.379888,14.999454,7.309503,6.073735,5.652989
2,22w6cq,1268,2020-01-26 07:45:00,2020-02-09 06:15:00,15,91.195694,88.02,64.98,148.50,79.74,...,unsupported operand type(s) for +: 'Timestamp'...,18.658,11.386916,2.558825,0.056012,9.006795,11.668639,5.786399,5.491401,4.804728
3,244bwh,1306,2020-08-31 10:45:00,2020-09-14 09:00:00,15,98.060995,96.30,57.60,158.94,90.90,...,unsupported operand type(s) for +: 'Timestamp'...,8.195,11.868266,1.043974,0.045043,5.430712,10.257157,4.177066,5.655619,5.043937
4,2857fm,1261,2019-10-01 05:45:00,2019-10-14 20:45:00,14,99.293481,98.46,68.04,146.52,91.08,...,unsupported operand type(s) for +: 'Timestamp'...,8.577,12.273983,1.029788,0.026504,5.694902,10.276845,3.543256,5.685100,5.086881


In [8]:
#metrics_df.reset_index().to_csv(CGM_METRICS_PATH)

In [9]:
metrics_df.to_csv(CGM_METRICS_PATH)